# 99 — M6 CERTIFIED 永続カタログ

persist 上の **既存** M6 `CERTIFIED` を全件スキャンし、
`campaign_b/_m6_certified_catalog/CATALOG.json` にマージする。

- 再実行で全件再スキャン + 新規追加
- kernel 再起動後もディスク上のカタログが残る
- CERTIFIED を捏造しない（レポートが主張したものだけ）
- Campaign B の `claim_scope` は `SCREENING_ONLY` のままになり得る


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'm6_certified_catalog.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/m6_certified_catalog.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.m6_certified_catalog import scan_and_update_catalog, catalog_path

validate_persistent_root()
RESULT = scan_and_update_catalog(PERSIST_ROOT)
print('catalog', catalog_path(PERSIST_ROOT))
print(json.dumps({
    'scanned_at': RESULT.get('scanned_at'),
    'total': RESULT.get('total'),
    'newly_found_count': len(RESULT.get('newly_found') or []),
    'certification_status': RESULT.get('certification_status'),
    'claim_scope': RESULT.get('claim_scope'),
}, indent=2))


In [ ]:
print('=== newly found ===')
print(json.dumps(RESULT.get('newly_found') or [], indent=2, ensure_ascii=False, default=str))
print('=== all certified ===')
print(json.dumps(RESULT.get('all_certified') or [], indent=2, ensure_ascii=False, default=str))
